In [2]:
import os
import pickle
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)

print("TensorFlow Version :", tf.__version__)

TensorFlow Version : 2.16.1


In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42
INITIAL_EPOCHS = 30

dataset_path = "../../datasets/food-101/images"

print(dataset_path)
print(os.path.exists(dataset_path))

../../datasets/food-101/images
True


In [4]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 101000 files belonging to 101 classes.
Using 80800 files for training.


In [5]:
validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 101000 files belonging to 101 classes.
Using 20200 files for validation.


In [6]:
class_names = train_dataset.class_names

print("Total Classes :", len(class_names))
print(class_names[:10])

Total Classes : 101
['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito']


In [7]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(1)
validation_dataset = validation_dataset.prefetch(1)

print("Dataset Ready")

Dataset Ready


In [8]:
print(type(train_dataset))

for images, labels in train_dataset.take(1):
    print(images.shape)
    print(labels.shape)

<class 'tensorflow.python.data.ops.prefetch_op._PrefetchDataset'>
(16, 224, 224, 3)
(16,)


In [9]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
], name="data_augmentation")

In [10]:
base_model = EfficientNetB0(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

print(len(base_model.layers))

238


In [11]:
inputs = tf.keras.Input(shape=(224,224,3))

x = data_augmentation(inputs)

x = tf.keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.4)(x)

outputs = layers.Dense(
    101,
    activation="softmax"
)(x)

model = Model(inputs, outputs)

print("Model Created")

Model Created


In [12]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Compiled")

Compiled


In [13]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 101)            │       129,381 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,178,952 (15.94 MB)

 Trainable params: 129,381 (505.39 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [14]:
checkpoint = ModelCheckpoint(
    "../saved_models/efficientnetb0_initial.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

csv_logger = CSVLogger(
    "../saved_models/training_log.csv"
)

print("Callbacks Ready")

Callbacks Ready


In [47]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=INITIAL_EPOCHS,
    callbacks=[
        checkpoint,
        early_stop,
        reduce_lr,
        csv_logger
    ],
    verbose=1
)

Epoch 1/30
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 0s 633ms/step - accuracy: 0.3812 - loss: 2.6363
Epoch 1: val_accuracy improved from None to 0.61218, saving model to ../saved_models/efficientnetb0_initial.keras

Epoch 1: finished saving model to ../saved_models/efficientnetb0_initial.keras
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 3873s 765ms/step - accuracy: 0.4596 - loss: 2.2033 - val_accuracy: 0.6122 - val_loss: 1.4765 - learning_rate: 0.0010
Epoch 2/30
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 0s 451ms/step - accuracy: 0.5289 - loss: 1.8629
Epoch 2: val_accuracy improved from 0.61218 to 0.63163, saving model to ../saved_models/efficientnetb0_initial.keras

Epoch 2: finished saving model to ../saved_models/efficientnetb0_initial.keras
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 2764s 547ms/step - accuracy: 0.5326 - loss: 1.8509 - val_accuracy: 0.6316 - val_loss: 1.4058 - learning_rate: 0.0010
Epoch 3/30
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5402 - loss: 1.8204
Epoch 3: val_accuracy improved from 0.63163 to

In [15]:
import json
import os

os.makedirs("saved_models", exist_ok=True)

with open("saved_models/class_names.json", "w") as f:
    json.dump(class_names, f)

print("class_names.json saved successfully.")

class_names.json saved successfully.


In [16]:
import os

print("Current Working Directory:")
print(os.getcwd())

print("\nFiles in saved_models:")
print(os.listdir("saved_models"))

Current Working Directory:
c:\Users\spavi\ICBT CAMPUS FOLDER\AI-Based-Food-Recognition-System\ai_model\notebooks

Files in saved_models:
['class_names.json']
